In [1]:
import os
from git import Repo
from langchain_text_splitters import Language,RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.memory import ConversationSummaryMemory
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.history_aware_retriever import create_history_aware_retriever



/Users/apple/Desktop/Source_Code_Analyser/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/d_/tv2cwk6x1w9dfjyttj4kppxh0000gn/T/ipykernel_1235/3006665593.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.generic import GenericLoader


In [ ]:
from git import Repo
repo = Repo.clone_from("https://github.com/RajRajputGit/etl_project",to_path="../repo_sample")

GitCommandError: Cmd('git') failed due to: exit code(128)
  cmdline: git clone -v -- https://github.com/RajRajputGit/etl_project ../repo_sample
  stderr: 'fatal: destination path '../repo_sample' already exists and is not an empty directory.
'

In [3]:
from langchain_community.document_loaders.parsers.language.language_parser import Language
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.document_loaders.generic import GenericLoader
parser = LanguageParser(language="python",parser_threshold=500)
loader = GenericLoader.from_filesystem("../repo_sample",
                glob = "**/*",  
                suffixes = [".py"],
                parser=parser)

In [4]:
documents = loader.load()

In [5]:
len(documents)

3

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
document_splitter = RecursiveCharacterTextSplitter.from_language(language = "python",
                                                chunk_size= 500,
                                                chunk_overlap = 20)

In [7]:
splitted_docs = document_splitter.split_documents(documents)

In [8]:
len(splitted_docs)

31

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [10]:
from langchain_chroma import Chroma
vecotordb = Chroma.from_documents(splitted_docs,embedding=embeddings,persist_directory="../vectordata")

In [11]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = api_key
llm = ChatGroq(api_key=api_key,model="llama3-8b-8192")

In [ ]:
retriever = vecotordb.as_retriever()